In [0]:
import json

# ============================================================
# Source Onboarding Agent tools.
# The agent proposes; these tools check, test, ask the human and only then act.
# All network/storage work is delegated to _utils_bronze_framework.
# ============================================================


def _no_data(sample):
    return sample is None or sample.strip() in ("", "[]", "{}", "null")


def _dry_run_days(spec):
    """Smallest window that still produces at least one chunk (APIs with a delay need more days)."""
    incremental = spec["incremental"]
    offsets = [incremental.get("end_offset_days", 0)] + [
        ep.get("end_offset_days", incremental.get("end_offset_days", 0)) for ep in spec["endpoints"]
    ]
    return 1 + max(0, -min(offsets))


def propose_source_spec(spec_yaml):
    """Agent tool: parse -> validate -> dry run -> human approval -> save YAML -> initial load."""

    # 1. Parse YAML
    try:
        spec = load_source_spec(spec_yaml)
    except ValueError as e:
        print(f"🛡️ Spec rejected (YAML): {str(e)[:150]}")
        return json.dumps({
            "status": "rejected", "stage": "yaml", "errors": [str(e)],
            "instruction": "Fix the YAML syntax. Quote values that start with { or %, or that contain ': '.",
        })

    # 2. Validate rules + name not already taken
    errors = validate_source_spec(spec)
    if spec.get("source_name") in list_source_specs():
        errors.append(f"source '{spec['source_name']}' already exists; choose another source_name or ask the user")
    if errors:
        print(f"🛡️ Spec rejected by validation: {errors}")
        return json.dumps({
            "status": "rejected", "stage": "validation", "errors": errors,
            "instruction": "Change the spec according to the errors. Do not resend the same spec.",
        })

    # 3. Dry run: the API must answer AND return data
    try:
        summary = run_source_spec(spec, days_back_override=_dry_run_days(spec), dry_run=True)
    except Exception as e:
        return json.dumps({"status": "error", "stage": "dry_run", "errors": [str(e)[:500]]})

    problems = [f"{r['endpoint_name']} {r['chunk_start']}->{r['chunk_end']}: {r['error_message']}"
                for r in summary["results"] if r["status"] == "error"]
    problems += [f"{r['endpoint_name']}: the API answered but returned no data; check URL, params and dates"
                 for r in summary["results"] if r["status"] != "error" and _no_data(r["sample"])]
    if not summary["results"]:
        problems.append("dry run produced no chunks; check days_back and end_offset_days")
    if problems:
        print(f"🧪 Dry run failed, the agent will fix it: {problems}")
        return json.dumps({
            "status": "error", "stage": "dry_run", "errors": problems,
            "instruction": "Probe the endpoint again if needed, fix the spec and propose it again.",
        })

    # 4. Human approval: only for specs that already work
    print("\n===== THE AGENT PROPOSES A NEW SOURCE =====\n")
    print(spec_to_yaml(spec))
    shown = set()
    for r in summary["results"]:
        if r["endpoint_name"] not in shown:
            shown.add(r["endpoint_name"])
            print(f"🧪 Dry run sample · {r['endpoint_name']} ({r['chunk_start']}->{r['chunk_end']}):")
            print(r["sample"][:400], "\n")

    if input("Approve this source? (y/n): ").strip().lower() != "y":
        return json.dumps({"status": "rejected_by_human",
                           "message": "The user did not approve the new source. Tell the user."})

    # 5. Save the spec (versioned in Git) and run the initial load
    path = save_source_spec_file(spec)
    print(f"\n💾 Spec saved: {path}\n")
    load = run_source_spec(spec)

    print(f"\n✅ Source {spec['source_name']} onboarded · {load['chunks_ok']} ok / {load['chunks_error']} errors")
    return json.dumps({
        "status": "created",
        "source_name": spec["source_name"],
        "spec_file": path,
        "chunks_ok": load["chunks_ok"],
        "chunks_error": load["chunks_error"],
        "bronze_tables": list(load["bronze_tables"].keys()),
    })